# Daily Challenge: LangChain Pipelines with Open-Source LLMs (Student)
Use this guided notebook with TODOs. Runs on CPU with small HF models (e.g., flan-t5-small).

## What you'll learn
- Set up LangChain with lightweight open-source models.
- Build an LLMChain using a prompt template.
- Compose a two-step Runnable pipeline (summary ? bullets).
- Bonus: add a simple conversation chain with memory.

## What you will create
- Installed environment for LangChain + transformers.
- LLMChain that rewrites text in a simpler style.
- Runnable pipeline that summarizes then bullet-izes text.
- (Bonus) Conversation chain showing memory.

## Part 1: Environment setup (fast)
Install needed packages. CPU is fine for tiny models.

In [ ]:
!nvidia-smi || echo "CPU runtime"

In [ ]:
pip install -q "langchain==0.1.7" "langchain-community==0.0.20" "transformers==4.37.2" "sentencepiece" "accelerate"

## Part 2: Load a tiny model and build your first LLMChain
Use a small model (e.g., google/flan-t5-small) to keep inference quick.

In [ ]:

# TODO: import libs
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain import PromptTemplate, LLMChain


In [ ]:

# TODO: choose a small model
model_name = "google/flan-t5-small"  # keep small for CPU


In [ ]:

# TODO: load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


In [ ]:

# TODO: create a generation pipeline
gen_pipeline = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
)
llm = HuggingFacePipeline(pipeline=gen_pipeline)


In [ ]:

# TODO: build prompt + LLMChain for friendly rewriting
template = "Rewrite this text to be simpler for beginners:{text}"
prompt = PromptTemplate(template=template, input_variables=["text"])
chain = LLMChain(prompt=prompt, llm=llm)

sample_text = "LangChain helps you build LLM apps by composing prompts, models, and tools."
rewritten = chain.run(text=sample_text)
print(rewritten)


## Part 3: Two-step pipeline (summary ? bullets)
Summarize a paragraph, then turn it into 3 bullets using the same LLM.

In [ ]:
from langchain.schema.runnable import RunnableLambda  # if needed, depending on version
from langchain import PromptTemplate

#To-Do define run templates
summary_prompt = PromptTemplate(
    template="Summarize the following paragraph in a concise way: {paragraph}",
    input_variables=["paragraph"],
)
bullets_prompt = PromptTemplate(
    template="Turn the following summary into 3 bullet points: {summary}",
    input_variables=["summary"],
)

In [ ]:
# First stage: paragraph -> summary (string)
summary_chain = summary_prompt | llm

# Full chain:
# 1. Take input {"paragraph": ...}
# 2. Run summary_chain to get a summary string
# 3. Wrap into {"summary": summary}
# 4. Run bullets_prompt, then llm
summarize_then_bullets = (
    {"summary": summary_chain}   # this creates a dict runnable
    | bullets_prompt
    | llm
)

In [ ]:
paragraph = """LangChain is a framework for building applications with large language models by composing prompts, models, and tools. It supports chains, agents, and retrieval workflows."""
bullets_output = summarize_then_bullets.invoke({"paragraph": paragraph})
print(bullets_output)


## Part 4 (Bonus): Conversation chain with memory
Show how two turns keep context.

In [ ]:
# build a simple conversation chain
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()

# Original conversation without system style
print("--- Original Conversation ---")
convo_original = ConversationChain(llm=llm, memory=memory, verbose=False)
reply1_original = convo_original.predict(input="Hi there! What's LangChain?")
reply2_original = convo_original.predict(input="Can it help me build a simple chatbot?")
print("Turn 1 (Original):", reply1_original)
print("Turn 2 (Original):", reply2_original)

# Conversation with system style
print("\n--- Conversation with System Style ---")
# Reset memory for the new conversation style demonstration
memory_styled = ConversationBufferMemory()

# Add a system message to the prompt for the styled conversation
styled_template = "The following is a friendly conversation between a human and an AI. The AI is concise and encouraging.\n\nCurrent conversation:\n{history}\nHuman: {input}\nAI:"
styled_prompt = PromptTemplate(input_variables=["history", "input"], template=styled_template)

convo_styled = ConversationChain(llm=llm, memory=memory_styled, prompt=styled_prompt, verbose=False)
reply1_styled = convo_styled.predict(input="Hello, AI! Tell me about LangChain.")
reply2_styled = convo_styled.predict(input="That sounds great. Can I use it for sentiment analysis?")
print("Turn 1 (Styled):", reply1_styled)
print("Turn 2 (Styled):", reply2_styled)

## Your observations (fill in)
- Latency: The small T5 model is quite fast on CPU, responses are nearly instantaneous.
- Quality: The rewriting and summarization are decent for a small model, but sometimes lack nuance. The bullet points are generally accurate.
- Quirks: Occasionally, the model might repeat phrases or generate slightly irrelevant content, especially when the input is complex. The conversational memory works, but responses can still be a bit generic.